### About Dataset
##### **Context**
**Predict behavior to retain customers. You can analyze all relevant customer data and develop focused customer retention programs.** [IBM Sample Data Sets]
#####  **Content**
Each row represents a customer, each column contains customer’s attributes described on the column Metadata.

The data set includes information about:

- **Customers who left within the last month** : the column is called Churn
- **Services that each customer has signed up for** : phone, multiple lines, internet, online security, online backup, device protection, tech support, and - streaming TV and movies
- **Customer account information** : how long they’ve been a customer, contract, payment method, paperless billing, monthly charges, and total charges
- **Demographic info about customers** : gender, age range, and if they have partners and dependents

## **LOADING THE DATASET**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../data/raw/Customer_Churn_Data.csv')
df.head()

# **EDA**

In [ ]:
df.shape

In [ ]:
# Ensure TotalCharges is numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

In [ ]:
# Drop Pure Identifier Columns

df = df.drop(columns=["customerID"], errors="ignore")
print(f"Remaining columns: {df.shape[1]}")


In [ ]:
df.info()

In [ ]:
df.describe(include = 'all')

In [ ]:
df.columns.to_list()

In [ ]:
df.isnull().sum()

## **Univariate Analysis**

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
df['Churn'].value_counts()

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x="Churn", hue="Churn", palette=["#2ecc71", "#e74c3c"], legend=False)
plt.title("Distribution of Target Variable (Churn)", fontsize=14, pad=10)
plt.xlabel("Churn Status")
plt.ylabel("Customer Count")

# Annotate counts and percentages
total = len(df)
for p in ax.patches:
    count = int(p.get_height())
    pct = f"{100 * count / total:.1f}%"
    ax.annotate(f"{count}\n({pct})", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=11)
plt.show()


In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in num_cols:
    fig, (ax_box, ax_hist) = plt.subplots(
        nrows=2, 
        sharex=True, 
        figsize=(10, 6), 
        gridspec_kw={"height_ratios": [0.25, 0.75]}
    )
    
    # Outliers / Spread
    sns.boxplot(data=df, x=col, ax=ax_box, color="#3498db", fliersize=3)
    ax_box.set(xlabel="")
    ax_box.set_title(f"Univariate Analysis: {col}", fontsize=14)
    
    # Distribution / KDE
    sns.histplot(data=df, x=col, kde=True, ax=ax_hist, color="#3498db", bins=30)
    ax_hist.set_ylabel("Count")
    
    plt.tight_layout()
    plt.show()


In [ ]:
demographic_cols = ["gender", "SeniorCitizen", "Partner", "Dependents"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(demographic_cols):
    ax = axes[i]
    sns.countplot(data=df, x=col, ax=ax, hue=col, palette="Blues_r", legend=False)
    ax.set_title(f"Distribution: {col}", fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    
    # Add percentage labels
    for p in ax.patches:
        height = p.get_height()
        pct = f"{100 * height / len(df):.1f}%"
        ax.annotate(pct, (p.get_x() + p.get_width() / 2., height),
                    ha="center", va="bottom", fontsize=10, xytext=(0, 2), textcoords="offset points")

plt.tight_layout()
plt.show()


In [ ]:
service_cols = [
    "PhoneService", "MultipleLines", "InternetService", 
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", 
    "TechSupport", "StreamingTV", "StreamingMovies"
]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(service_cols):
    ax = axes[i]
    sns.countplot(data=df, x=col, ax=ax, hue=col, palette="viridis", legend=False)
    ax.set_title(f"Service: {col}", fontsize=11)
    ax.set_xlabel("")
    ax.tick_params(axis='x', rotation=20)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()


In [ ]:
contract_cols = ["Contract", "PaperlessBilling", "PaymentMethod"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(contract_cols):
    ax = axes[i]
    sns.countplot(data=df, x=col, ax=ax, hue=col, palette="mako", legend=False)
    ax.set_title(f"Account: {col}", fontsize=12)
    ax.set_xlabel("")
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()


## **Bivariate Analysis**

In [ ]:
churn_palette = {"No": "#2ecc71", "Yes": "#e74c3c"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE distribution
sns.kdeplot(data=df, x="tenure", hue="Churn", common_norm=False, palette=churn_palette, fill=True, alpha=0.4, ax=axes[0])
axes[0].set_title("Tenure Distribution by Churn", fontsize=13)

# Boxplot
sns.boxplot(data=df, x="Churn", y="tenure", palette=churn_palette, ax=axes[1])
axes[1].set_title("Tenure Spread & Median by Churn", fontsize=13)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE distribution
sns.kdeplot(data=df, x="MonthlyCharges", hue="Churn", common_norm=False, palette=churn_palette, fill=True, alpha=0.4, ax=axes[0])
axes[0].set_title("Monthly Charges Distribution by Churn", fontsize=13)

# Boxplot
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", palette=churn_palette, ax=axes[1])
axes[1].set_title("Monthly Charges Spread by Churn", fontsize=13)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE distribution
sns.kdeplot(data=df, x="TotalCharges", hue="Churn", common_norm=False, palette=churn_palette, fill=True, alpha=0.4, ax=axes[0])
axes[0].set_title("Total Charges Distribution by Churn", fontsize=13)

# Boxplot
sns.boxplot(data=df, x="Churn", y="TotalCharges", palette=churn_palette, ax=axes[1])
axes[1].set_title("Total Charges Spread by Churn", fontsize=13)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.countplot(data=df, x="Contract", hue="Churn", palette=churn_palette)
plt.title("Churn by Contract Type", fontsize=14)
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")

# Display percentages above each bar
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{int(height)}", (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10, xytext=(0, 2), textcoords='offset points')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Internet Service vs Churn
sns.countplot(data=df, x="InternetService", hue="Churn", palette=churn_palette, ax=axes[0])
axes[0].set_title("Churn by Internet Service Type", fontsize=13)

# Tech Support vs Churn
sns.countplot(data=df, x="TechSupport", hue="Churn", palette=churn_palette, ax=axes[1])
axes[1].set_title("Churn by Tech Support Availability", fontsize=13)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.countplot(data=df, y="PaymentMethod", hue="Churn", palette=churn_palette)
plt.title("Churn by Payment Method", fontsize=14)
plt.xlabel("Number of Customers")
plt.ylabel("Payment Method")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

demographics = ["SeniorCitizen", "Partner", "Dependents"]

for i, col in enumerate(demographics):
    sns.countplot(data=df, x=col, hue="Churn", palette=churn_palette, ax=axes[i])
    axes[i].set_title(f"Churn by {col}", fontsize=13)
    axes[i].set_ylabel("Customer Count" if i == 0 else "")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="tenure",
    y="MonthlyCharges",
    hue="Churn",
    palette=churn_palette,
    alpha=0.6,
    s=40
)
plt.title("Tenure vs Monthly Charges (by Churn Status)", fontsize=14)
plt.xlabel("Tenure (Months)")
plt.ylabel("Monthly Charges ($)")
plt.show()


In [ ]:
# Create tenure bins (Year 1, Year 2, etc.)
tenure_bins = [0, 12, 24, 36, 48, 60, 72]
tenure_labels = ["0-1 Yr", "1-2 Yrs", "2-3 Yrs", "3-4 Yrs", "4-5 Yrs", "5-6 Yrs"]
df_cohort = df.copy()
df_cohort["Tenure_Group"] = pd.cut(df_cohort["tenure"], bins=tenure_bins, labels=tenure_labels, include_lowest=True)

# Calculate churn rate per cohort
cohort_churn = df_cohort.groupby("Tenure_Group", observed=False)["Churn"].apply(lambda x: (x == "Yes").mean() * 100).reset_index()
cohort_churn.rename(columns={"Churn": "Churn_Rate_Pct"}, inplace=True)

plt.figure(figsize=(9, 4))
ax = sns.barplot(data=cohort_churn, x="Tenure_Group", y="Churn_Rate_Pct", palette="Reds_r")
plt.title("Churn Rate (%) Across Tenure Cohorts", fontsize=13)
plt.ylabel("Churn Rate (%)")
plt.xlabel("Tenure Cohort")
plt.ylim(0, 60)

for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, xytext=(0, 3), textcoords='offset points')
plt.show()


## **Correlation between numeric columns**

In [ ]:
# Select numeric features + include numeric Churn flag for correlation with target
numeric_df = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']].copy()
numeric_df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Compute Pearson correlation matrix
corr_matrix = numeric_df.corr()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=-1,
    vmax=1,
    linewidths=0.8,
    square=True
)
plt.title("Full Correlation Matrix of Numeric Columns", fontsize=14, pad=15)
plt.tight_layout()
plt.show()


# **Data Cleaning and Feature Engineering**

In [ ]:
# Separate Features (X) from Target (y)

# Target variable
y = df["Churn"].map({"Yes": 1, "No": 0})

# Features dataframe
X = df.drop(columns=["Churn"])

print(f"Features shape (X): {X.shape}")
print(f"Target shape   (y): {y.shape}")
print("\nTarget class distribution:")
print(y.value_counts(normalize=True).apply(lambda x: f"{x:.2%}"))


In [ ]:
# Simplify redundant categories

replace_no_internet = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

for col in replace_no_internet:
    X[col] = X[col].replace({"No internet service": "No"})

X["MultipleLines"] = X["MultipleLines"].replace({"No phone service": "No"})

# Feature Engineering
X["AvgMonthlyCharges"] = X["TotalCharges"] / (X["tenure"] + 1)
X["TotalServices"] = (
    (X["PhoneService"] == "Yes").astype(int) +
    (X["MultipleLines"] == "Yes").astype(int) +
    (X["InternetService"] != "No").astype(int) +
    (X["OnlineSecurity"] == "Yes").astype(int) +
    (X["OnlineBackup"] == "Yes").astype(int) +
    (X["DeviceProtection"] == "Yes").astype(int) +
    (X["TechSupport"] == "Yes").astype(int) +
    (X["StreamingTV"] == "Yes").astype(int) +
    (X["StreamingMovies"] == "Yes").astype(int)
)
X["IsLongTermContract"] = (X["Contract"] != "Month-to-month").astype(int)

print(f"Features after engineering: {X.shape[1]}")


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}")
print(f"Train Churn Rate: {y_train.mean():.2%}, Test Churn Rate: {y_test.mean():.2%}")


In [ ]:
continuous_features = ["tenure", "MonthlyCharges", "TotalCharges", "AvgMonthlyCharges"]

for col in continuous_features:
    lower = X_train[col].quantile(0.01)
    upper = X_train[col].quantile(0.99)
    X_train[col] = X_train[col].clip(lower=lower, upper=upper)
    X_test[col] = X_test[col].clip(lower=lower, upper=upper)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder

# Group numerical and categorical columns
pt_features = ["TotalCharges", "AvgMonthlyCharges"]
standard_features = [c for c in X_train.select_dtypes(include=["number"]).columns if c not in pt_features]
categorical_cols = X_train.select_dtypes(include=["object", "str"]).columns.tolist()

# 1. Pipeline for numerical features to be PowerTransformed (Yeo-Johnson)
pt_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("skewness", PowerTransformer(method="yeo-johnson")),
    ("scale", StandardScaler())
])

# 2. Pipeline for other numerical features (only imputation and scaling)
standard_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

# 3. Pipeline for Categorical Columns
categorical_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Pure ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("pt_numerical", pt_pipeline, pt_features),
    ("standard_numerical", standard_pipeline, standard_features),
    ("categorical", categorical_pipeline, categorical_cols)
])

preprocessor


In [ ]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(class_weight="balanced", n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

results = []
pipelines = {}

header = f"{'Model':<22} | {'Accuracy':<8} | {'F1-Macro':<8} | {'Recall':<8} | {'ROC-AUC':<8}"
print(header)
print("-" * len(header))

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])
    pipelines[name] = pipe

    # cross_validate computes all metrics in a single pass without redundant fits
    cv_res = cross_validate(
        pipe, X_train, y_train, cv=3,
        scoring=["accuracy", "f1_macro", "recall", "roc_auc"],
        n_jobs=1
    )
    acc = cv_res["test_accuracy"].mean()
    f1 = cv_res["test_f1_macro"].mean()
    rec = cv_res["test_recall"].mean()
    auc = cv_res["test_roc_auc"].mean()

    results.append({"Model": name, "Accuracy": acc, "F1_Macro": f1, "Recall": rec, "ROC_AUC": auc})
    print(f"{name:<22} | {acc:<8.3f} | {f1:<8.3f} | {rec:<8.3f} | {auc:<8.3f}")

# Compare models and select champion by F1-Macro
results_df = pd.DataFrame(results).sort_values(by="F1_Macro", ascending=False)
best_model_name = results_df.iloc[0]["Model"]
best_pipe = pipelines[best_model_name]

print("\n" + "=" * 65)
print(f"CHAMPION MODEL: {best_model_name} (F1-Macro: {results_df.iloc[0]['F1_Macro']:.4f})")
print("Fitting champion pipeline on full training data...")
best_pipe.fit(X_train, y_train)

# Unseen test evaluation
test_acc = best_pipe.score(X_test, y_test)
print(f"Unseen Test Set Accuracy: {test_acc:.4f}")

# Save final model artifact
os.makedirs("../models", exist_ok=True)
artifact_path = "../models/best_model_pipeline.joblib"
joblib.dump(best_pipe, artifact_path, compress=3)
print(f"Saved final model artifact to {artifact_path} ({os.path.getsize(artifact_path) / 1024:.1f} KB)")
print("=" * 65)
